# Travel Agent Notebook

In [16]:
import os
from dotenv import load_dotenv

load_dotenv()

True

## Tools

In [17]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> str:
    """Search the web for information"""
    return tavily_client.search(query)

@tool
def update_trip_info(
    runtime: ToolRuntime,
    destination: str | None = None,
    season: str | None = None,
    departure_date: str | None = None,
    return_date: str | None = None,
    budget: str | None = None
    ) -> Command:
    """Save any trip details the user has provided. Pass only the fields
    the user mentioned this turn; leave the rest as None."""

    updates = {k: v for k, v in {
        "destination": destination,
        "season": season,
        "departure_date": departure_date,
        "return_date": return_date,
        "budget": budget
    }.items() if v is not None}
    
    return Command(update=updates)


## Custom State

In [18]:
from langchain.agents import AgentState

class TripState(AgentState):
    destination: str
    season: str
    departure_date: str
    return_date: str
    budget: str

## System Prompt

In [19]:
system_prompt = """
You are an expert travel agent. Given the user's travel preferences and requirements,  
find the best potential travel destinations and activities as well as provide tips and 
helpful information when appropriate.

Use the web_search tool to find up to date information on potential destinations. 

When a user gives a trip detail (destination, season, dates, budget), call update_trip_info
to save it before replying.

"""

## Agent

In [20]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver  

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search],
    system_prompt=system_prompt,
    checkpointer=InMemorySaver()
)

In [21]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Find a place to visit in Europe during December")

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Find a place to visit in Europe during December")
        ]},
    {"configurable": {"thread_id": "1"}}
)

In [15]:
print(response["messages"][-1].content)

Here are three well-rounded European options for a December visit, each with a few key activities:

1) Prague, Czech Republic
- Why December: Fairy-tale winter charm with twinkling Christmas markets and a compact, walkable old town.
- Activities:
  - Explore the Old Town Square Christmas Market and try local pastries and mulled wine.
  - See Prague Castle, St. Vitus Cathedral, and Charles Bridge dusted in snow.
  - Take a river cruise on the Vltava or enjoy a cozy café crawl in Malá Strana.
  - Ice skate at an outdoor rink and enjoy hearty Czech cuisine plus classical concerts.
  - Optional day trip: Kutná Hora or Karlštejn Castle.

2) Tromsø, Norway
- Why December: Premier Arctic destination for Northern Lights, snow-covered landscapes, and authentic winter experiences.
- Activities:
  - Northern Lights tours and night sky viewing (clear, dark nights boost chances).
  - Dog sledding or snowmobile safaris through pristine Arctic terrain.
  - Sami culture experiences and reindeer sleddi